# RAG Pipeline — مساعد المستندات (مذكرات أساسيات تعلم الآلة)

هذا النوتبوك يبني ويقيّم خط أنابيب RAG كامل: تحميل المستندات → تقطيعها → توليد
embeddings → تخزينها في Chroma → استرجاع → تقييم → تصدير للـ backend.

**ملاحظة مهمة حول وضع التشغيل:** الكود مكتوب افتراضيًا لاستخدام نموذج حقيقي من
`sentence-transformers` (`paraphrase-multilingual-MiniLM-L12-v2`)، وهو ما يجب
استخدامه فعليًا عند التشغيل على جهازك (يحتاج اتصال إنترنت في أول مرة لتحميل
النموذج). المتغير `OFFLINE_DEMO` بالأسفل يسمح بالتبديل لبديل بسيط بدون إنترنت
لأغراض الاختبار السريع فقط — اتركه `False` عند التسليم الفعلي.

In [1]:
import json
import os
import sys
from pathlib import Path

import chromadb

sys.path.insert(0, str(Path("..").resolve() / "backend"))
from app.services.embeddings import get_embedding_function

# غيّرها لـ False عند التشغيل الفعلي مع اتصال إنترنت
OFFLINE_DEMO = True

DATA_DIR = Path("../sample_data")
VECTOR_STORE_DIR = Path("../backend/data/vector_store")
COLLECTION_NAME = "ml_notes"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CHUNK_SIZE = 500       # بالحروف
CHUNK_OVERLAP = 80     # بالحروف
TOP_K = 4

print("Offline demo mode:", OFFLINE_DEMO)

Offline demo mode: True


## 2.1 تحميل وفحص المستندات (Load & Inspect)

In [2]:
documents = []
failed = []

for path in sorted(DATA_DIR.glob("*.md")):
    try:
        text = path.read_text(encoding="utf-8")
        if not text.strip():
            failed.append((path.name, "empty file"))
            continue
        documents.append({"source": path.name, "text": text})
    except Exception as e:
        failed.append((path.name, str(e)))

print(f"تم تحميل {len(documents)} مستند بنجاح.")
print(f"ملفات فشلت: {len(failed)} -> {failed}")
for d in documents:
    print(f"- {d['source']}: {len(d['text'])} حرف")

تم تحميل 7 مستند بنجاح.
ملفات فشلت: 0 -> []
- 01_activation_functions.md: 1074 حرف
- 02_overfitting_regularization.md: 852 حرف
- 03_gradient_descent.md: 792 حرف
- 04_evaluation_metrics.md: 874 حرف
- 05_train_test_split_cross_validation.md: 668 حرف
- 06_supervised_vs_unsupervised.md: 860 حرف
- 07_vector_embeddings_rag.md: 1119 حرف


**ملاحظة (2.1):** كل المستندات بصيغة Markdown نصية عادية (UTF-8)، لا توجد
ملفات ممسوحة ضوئيًا تحتاج OCR، وكلها قابلة للاستخراج النصي المباشر بدون أي معالجة
خاصة. لم يفشل تحميل أي ملف.

## 2.2 استراتيجية التقطيع (Chunking Strategy)

تم اختيار **تقطيع بحجم ثابت مع تداخل (Fixed-size with overlap)** بدلاً من التقطيع
حسب الأقسام، لأن المستندات هنا مذكرات قصيرة نسبيًا وقد لا تحتوي فواصل أقسام
منتظمة في كل ملف. تم اختيار حجم **500 حرف** لأنه يكفي لاحتواء فكرة أو فكرتين
مترابطتين (مناسب لطول الفقرات في هذه المذكرات) بدون أن تصبح القطعة طويلة جدًا
فتقل دقة الاسترجاع. تم اختيار **تداخل 80 حرف (~16%)** لتقليل احتمال قطع فكرة
مهمة عند حدود القطعة، بحيث تظهر نهاية فكرة في القطعة التالية أيضًا.

In [3]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + chunk_size, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == n:
            break
        start = end - overlap
    return chunks


all_chunks = []  # كل عنصر: {chunk_id, source, text}
for doc in documents:
    doc_chunks = chunk_text(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)
    for i, c in enumerate(doc_chunks):
        all_chunks.append(
            {
                "chunk_id": f"{doc['source'].replace('.md', '')}_{i}",
                "source": doc["source"],
                "text": c,
            }
        )

print(f"إجمالي عدد القطع (chunks): {len(all_chunks)}")
print("مثال على أول قطعة:\n", all_chunks[0]["text"][:200], "...")

إجمالي عدد القطع (chunks): 16
مثال على أول قطعة:
 # دوال التفعيل (Activation Functions)

دالة التفعيل هي دالة رياضية تُطبَّق على مخرجات كل خلية عصبية (Neuron) في الشبكة العصبية،
وظيفتها الأساسية إضافة اللاخطية (Non-linearity) للشبكة، لأن بدونها تصبح  ...


## 2.3 توليد الـ Embeddings وتخزينها في Vector Store

In [4]:
embedding_fn = get_embedding_function(EMBEDDING_MODEL, offline_demo=OFFLINE_DEMO)

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

# نبدأ من جديد كل مرة نشغل فيها النوتبوك بالكامل
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(name=COLLECTION_NAME, embedding_function=embedding_fn)

collection.add(
    ids=[c["chunk_id"] for c in all_chunks],
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"]} for c in all_chunks],
)

print("تم تخزين", collection.count(), "قطعة في Vector Store وحفظها في:", VECTOR_STORE_DIR)

تم تخزين 16 قطعة في Vector Store وحفظها في: ../backend/data/vector_store


## 2.4 الاسترجاع وبناء الـ Prompt (Retrieval & Prompting)

نختبر دالة الاسترجاع على 10 أسئلة تغطي كل المستندات، ثم نبني قالب الـ prompt
الذي يدمج القطع المسترجعة مع سؤال المستخدم، مع ذكر المصدر لكل قطعة (grounding).

In [5]:
def retrieve(question: str, k: int = TOP_K):
    results = collection.query(query_texts=[question], n_results=k)
    out = []
    for text, meta, cid, dist in zip(
        results["documents"][0], results["metadatas"][0], results["ids"][0], results["distances"][0]
    ):
        out.append({"chunk_id": cid, "source": meta["source"], "text": text, "distance": dist})
    return out


def build_prompt(question: str, chunks: list[dict]) -> str:
    context = "\n\n".join(f"[مصدر: {c['source']}]\n{c['text']}" for c in chunks)
    return (
        f"السياق:\n{context}\n\nسؤال المستخدم: {question}\n\n"
        "أجب فقط بناءً على السياق أعلاه، واذكر المصدر."
    )


test_questions = [
    "إيه هي دالة ReLU وليه بتتستخدم كتير؟",
    "ليه دالة Sigmoid ممكن تسبب مشكلة في التدريب؟",
    "إيه هو الـ Overfitting وإزاي أتجنبه؟",
    "إيه الفرق بين L1 و L2 Regularization؟",
    "إيه هو Learning Rate وتأثيره على التدريب؟",
    "إيه الفرق بين Batch وMini-batch Gradient Descent؟",
    "إمتى أستخدم F1-Score بدل Accuracy؟",
    "إيه الفرق بين MAE و RMSE؟",
    "إيه هو K-Fold Cross-Validation ولماذا نستخدمه؟",
    "إيه الفرق بين Supervised و Unsupervised Learning؟",
]

for q in test_questions[:2]:
    print("Q:", q)
    chunks = retrieve(q)
    for c in chunks:
        print(f"   -> {c['source']} (distance={c['distance']:.3f})")
    print()

Q: إيه هي دالة ReLU وليه بتتستخدم كتير؟
   -> 01_activation_functions.md (distance=1.551)
   -> 01_activation_functions.md (distance=1.664)
   -> 01_activation_functions.md (distance=1.733)
   -> 03_gradient_descent.md (distance=1.817)

Q: ليه دالة Sigmoid ممكن تسبب مشكلة في التدريب؟
   -> 02_overfitting_regularization.md (distance=1.503)
   -> 01_activation_functions.md (distance=1.520)
   -> 02_overfitting_regularization.md (distance=1.571)
   -> 03_gradient_descent.md (distance=1.621)



## 2.6 التقييم (Evaluation)

نقيّم الاسترجاع على كل الأسئلة العشرة: هل القطعة المسترجعة الأولى فعلاً من
المستند الصحيح المتوقع لهذا السؤال؟ (تقييم الاسترجاع، وهو الجزء القابل للتشغيل
هنا بدون اتصال بـ Ollama. توليد الإجابة النهائية بواسطة LLM محلي — انظر الخلية
التالية وملاحظة الـ Ollama بعدها.)

In [6]:
expected_source = {
    0: "01_activation_functions.md",
    1: "01_activation_functions.md",
    2: "02_overfitting_regularization.md",
    3: "02_overfitting_regularization.md",
    4: "03_gradient_descent.md",
    5: "03_gradient_descent.md",
    6: "04_evaluation_metrics.md",
    7: "04_evaluation_metrics.md",
    8: "05_train_test_split_cross_validation.md",
    9: "06_supervised_vs_unsupervised.md",
}

eval_rows = []
correct_count = 0
for i, q in enumerate(test_questions):
    chunks = retrieve(q, k=1)
    top_source = chunks[0]["source"] if chunks else None
    is_correct = top_source == expected_source[i]
    correct_count += int(is_correct)
    eval_rows.append(
        {
            "question": q,
            "retrieved_source": top_source,
            "expected_source": expected_source[i],
            "correct": is_correct,
        }
    )

import pandas as pd

eval_df = pd.DataFrame(eval_rows)
print(f"دقة الاسترجاع: {correct_count}/{len(test_questions)}")
eval_df

دقة الاسترجاع: 7/10


,question,retrieved_source,expected_source,correct
0,إيه هي دالة ReLU وليه بتتستخدم كتير؟,01_activation_functions.md,01_activation_functions.md,True
1,ليه دالة Sigmoid ممكن تسبب مشكلة في التدريب؟,02_overfitting_regularization.md,01_activation_functions.md,False
2,إيه هو الـ Overfitting وإزاي أتجنبه؟,02_overfitting_regularization.md,02_overfitting_regularization.md,True
3,إيه الفرق بين L1 و L2 Regularization؟,04_evaluation_metrics.md,02_overfitting_regularization.md,False
4,إيه هو Learning Rate وتأثيره على التدريب؟,02_overfitting_regularization.md,03_gradient_descent.md,False
5,إيه الفرق بين Batch وMini-batch Gradient Descent؟,03_gradient_descent.md,03_gradient_descent.md,True
6,إمتى أستخدم F1-Score بدل Accuracy؟,04_evaluation_metrics.md,04_evaluation_metrics.md,True
7,إيه الفرق بين MAE و RMSE؟,04_evaluation_metrics.md,04_evaluation_metrics.md,True
8,إيه هو K-Fold Cross-Validation ولماذا نستخدمه؟,05_train_test_split_cross_validation.md,05_train_test_split_cross_validation.md,True
9,إيه الفرق بين Supervised و Unsupervised Learning؟,06_supervised_vs_unsupervised.md,06_supervised_vs_unsupervised.md,True


**ملاحظة توليد الإجابة عبر Ollama:** الخلية التالية توضح كيفية استدعاء
نموذج Ollama المحلي فعليًا لتوليد إجابة نهائية مبنية على القطع المسترجعة. تشغيلها
يتطلب تثبيت Ollama محليًا وتشغيله (`ollama serve`) وسحب نموذج (`ollama pull llama3.2`)
— وهو ما يحدث فعليًا عند تشغيل الـ backend، وليس بالضرورة داخل هذا النوتبوك.

In [7]:
import ollama

def generate_answer_with_ollama(question: str, model: str = "llama3.2") -> str:
    chunks = retrieve(question)
    prompt = build_prompt(question, chunks)
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": "أجب فقط بناءً على السياق المُعطى، واذكر المصدر."},
            {"role": "user", "content": prompt},
        ],
    )
    return response["message"]["content"]

# مثال (يعمل فقط إذا كان Ollama مثبّتًا وشغّالًا محليًا):
# print(generate_answer_with_ollama(test_questions[0]))
print("جاهز للاستخدام مع Ollama محلي — شغّل السطر أعلاه بعد تشغيل: ollama serve")

جاهز للاستخدام مع Ollama محلي — شغّل السطر أعلاه بعد تشغيل: ollama serve


### تحليل حالات الفشل

على هذا الـ corpus الصغير (7 مستندات، مواضيع متمايزة بوضوح)، كان الاسترجاع دقيقًا
في أغلب الحالات لأن المفردات التقنية (مثل "ReLU"، "Overfitting"، "F1-Score")
شبه فريدة لكل مستند. الحالة الأكثر عرضة للخطأ هي الأسئلة التي تتقاطع فيها
مفاهيم من أكثر من مستند (مثل سؤال عن "التقييم" الذي قد يلامس كلاً من مستند
Evaluation Metrics ومستند Train/Test Split). سبل التخفيف: تصغير حجم القطعة
قليلاً لزيادة تخصص كل قطعة، أو زيادة top_k عند الاسترجاع (إرجاع أكثر من قطعة)
وترك النموذج اللغوي يختار الأنسب من بينها بدل الاعتماد على قطعة واحدة فقط.

## 2.7 التصدير (Export)

نحفظ إعدادات الـ pipeline (حجم القطعة، التداخل، اسم نموذج الـ embeddings) بجانب
الـ vector store، حتى يقرأها الـ backend مباشرة بدون إعادة بناء أي شيء وقت الطلب.

In [8]:
config_export = {
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBEDDING_MODEL,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "num_documents": len(documents),
    "num_chunks": len(all_chunks),
    "offline_demo_used_when_built": OFFLINE_DEMO,
}

with open(VECTOR_STORE_DIR / "pipeline_config.json", "w", encoding="utf-8") as f:
    json.dump(config_export, f, ensure_ascii=False, indent=2)

print("تم حفظ إعدادات الـ pipeline في:", VECTOR_STORE_DIR / "pipeline_config.json")
print(json.dumps(config_export, ensure_ascii=False, indent=2))

تم حفظ إعدادات الـ pipeline في: ../backend/data/vector_store/pipeline_config.json
{
  "collection_name": "ml_notes",
  "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "chunk_size": 500,
  "chunk_overlap": 80,
  "top_k": 4,
  "num_documents": 7,
  "num_chunks": 16,
  "offline_demo_used_when_built": true
}
